In [12]:
import pandas as pd
import re
import spacy
import nltk
from nltk.corpus import stopwords

# Download once
nltk.download('stopwords')

# Load spaCy
nlp = spacy.load("en_core_web_sm")

# Stopwords
nltk_stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ravid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
df = pd.read_csv("customer_support_tickets.csv")

print("Shape:", df.shape)
df.head()

Shape: (8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [14]:
# Normalize column names
df.columns = df.columns.str.lower().str.strip()

df = df.rename(columns={
    "ticket description": "ticket_text",
    "ticket type": "category",
    "ticket priority": "priority"
})

# Check
print(df.columns)

Index(['ticket id', 'customer name', 'customer email', 'customer age',
       'customer gender', 'product purchased', 'date of purchase', 'category',
       'ticket subject', 'ticket_text', 'ticket status', 'resolution',
       'priority', 'ticket channel', 'first response time',
       'time to resolution', 'customer satisfaction rating'],
      dtype='object')


In [15]:
def advanced_nlp_clean(text):
    if not isinstance(text, str): 
        return ""
    
    text = re.sub(r'\{.*?\}', '', text.lower())
    text = re.sub(r'\b\d+\.\d+(\.\d+)?\b', '', text)
    text = re.sub(r'[\n\t\r]', ' ', text)

    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if token.is_alpha:
            # keep important negations
            if token.text in ["not", "no", "never"]:
                tokens.append(token.text)
            elif token.text not in nltk_stop_words:
                tokens.append(token.lemma_)
    
    return " ".join(tokens)

In [16]:
print("please wait this process take 1-2 min!!")

df["clean_text"] = df["ticket_text"].apply(advanced_nlp_clean)
df[["ticket_text", "clean_text"]].head()

,ticket_text,clean_text
0,I'm having an issue with the {product_purchase...,issue please assist billing zip code appreciat...
1,I'm having an issue with the {product_purchase...,issue please assist need change exist product ...
2,I'm facing a problem with my {product_purchase...,face problem not turn work fine yesterday resp...
3,I'm having an issue with the {product_purchase...,issue please assist problem interested love se...
4,I'm having an issue with the {product_purchase...,issue please assist note seller not responsibl...


In [17]:
df = df.dropna(subset=["clean_text", "category", "priority"])
df = df[df["clean_text"] != ""]

print("New shape:", df.shape)

New shape: (8469, 18)


In [18]:
df["category"] = df["category"].str.lower().str.strip()
df["priority"] = df["priority"].str.lower().str.strip()

print("Categories:\n", df["category"].unique())
print("\nPriorities:\n", df["priority"].unique())

Categories:
 ['technical issue' 'billing inquiry' 'cancellation request'
 'product inquiry' 'refund request']

Priorities:
 ['critical' 'low' 'high' 'medium']


In [19]:
print("Category distribution:\n")
print(df["category"].value_counts())

print("\nPriority distribution:\n")
print(df["priority"].value_counts())

Category distribution:

category
refund request          1752
technical issue         1747
cancellation request    1695
product inquiry         1641
billing inquiry         1634
Name: count, dtype: int64

Priority distribution:

priority
medium      2192
critical    2129
high        2085
low         2063
Name: count, dtype: int64


In [20]:
df.to_csv("cleaned_ticket.csv", index=False)

print("✅ cleaned_ticket.csv saved!")

✅ cleaned_ticket.csv saved!


In [21]:
sample = "I am so angry! TV not working, want money back"
print("Cleaned:", advanced_nlp_clean(sample))

Cleaned: angry tv not work want money back
